<a href="https://colab.research.google.com/github/jeanbaptiste-billaud/mlops_lecture/blob/main/MLflow_Dagshub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From dataset to production: an end-to-end MLOps demo

Welcome! This notebook is a demonstration of the full lifecycle of a machine learning model: we start by getting a dataset and end by serving the trained model in production.

Along the way, you'll see that MLOps is a complex ecosystem, similar to an orchestra: each tool has its own role, but they only make music together.

| Step | What we do | Tool |
|---|---|---|
| 1. Get the data | Download the BUSI ultrasound dataset | DagsHub Data Engine |
| 2. Prepare the data | Explore, clean, deduplicate, split | pandas, scikit-learn |
| 3. Version the data | Track raw and processed data, push them to a remote | DVC + DagsHub |
| 4. Train and track | Train scikit-learn, XGBoost and PyTorch models, log params, metrics and models | MLflow + DagsHub |
| 5. Compare | Select the best run | MLflow |
| 6. Serve | Package the model and expose it as an API | BentoML |

Code lives on **GitHub**, while **DagsHub** hosts the data remote and the MLflow tracking server, so that every model can be traced back to the exact code and data that produced it.

# Setup

This section configures the whole toolchain once, in this order:

1. **GitHub**: fork and clone the project (code versioning)
2. **DagsHub**: connect your fork (remote storage + MLflow server)
3. **DagsHub authentication**: get a token, reused by DVC, MLflow and BentoML
4. **DVC**: point the data remote to your DagsHub repository
5. **MLflow**: select the experiment

- **If you're on Google Colab:** select a GPU runtime: *Runtime → Change runtime type → T4 GPU*

## 1. GitHub: fork and clone the project

### Install the GitHub CLI
Run **only** the cell matching your environment.

In [ ]:
%%bash
# GitHub CLI installation on Google Colab
mkdir -p -m 755 /etc/apt/keyrings
wget -qO- https://cli.github.com/packages/githubcli-archive-keyring.gpg > /etc/apt/keyrings/githubcli-archive-keyring.gpg
chmod go+r /etc/apt/keyrings/githubcli-archive-keyring.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" > /etc/apt/sources.list.d/github-cli.list
apt-get update -qq && apt-get install -y -qq gh

In [2]:
# Github CLI installation on Ubuntu WSL
!sudo apt install -y extrepo
!sudo extrepo enable github-cli
!sudo apt update
!sudo apt install -y gh

Installing:                     
  extrepo

Installing dependencies:
  extrepo-offline-data    libio-html-perl
  libauthen-sasl-perl     libio-socket-ssl-perl
  libclone-perl           libipc-system-simple-perl
  libcommon-sense-perl    libjson-perl
  libcrypt-urandom-perl   libjson-xs-perl
  libcryptx-perl          liblwp-mediatypes-perl
  libdata-dump-perl       liblwp-protocol-https-perl
  libencode-locale-perl   libmailtools-perl
  libfile-listing-perl    libnet-http-perl
  libfont-afm-perl        libnet-smtp-ssl-perl
  libhtml-form-perl       libnet-ssleay-perl
  libhtml-format-perl     libtimedate-perl
  libhtml-parser-perl     libtry-tiny-perl
  libhtml-tagset-perl     libtypes-serialiser-perl
  libhtml-tree-perl       liburi-perl
  libhttp-cookies-perl    libwww-perl
  libhttp-daemon-perl     libwww-robotrules-perl
  libhttp-date-perl       libyaml-libyaml-perl
  libhttp-message-perl    perl-openssl-defaults
  libhttp-negotiate-perl

Suggested packages:
  libdigest-hmac-perl   

### Authenticate to GitHub
Open a terminal (on Colab: *Terminal* button at the bottom left) and run:

```
gh auth login --hostname github.com --git-protocol https --web
```

Then run the next cell to check the login and let git use your GitHub credentials.

In [ ]:
!gh auth status
!gh auth setup-git

### Fork and clone
`gh repo fork` creates the fork **on your GitHub account**, and `--clone` also clones it locally:
`origin` points to your fork and `upstream` to the original repository.

In [ ]:
upstream_owner = 'jeanbaptiste-billaud'
repo_name = 'MLflow_Dagshub'

# Your GitHub username
gh_user = !gh api user --jq .login
gh_user = gh_user[0]

# Fork on GitHub and clone locally
!gh repo fork {upstream_owner}/{repo_name} --clone --default-branch-only
%cd {repo_name}

# Use the GitHub noreply address if your email is private
gh_id = !gh api user --jq .id
gh_email = !gh api user --jq .email
email = gh_email[0] if gh_email[0] != "null" else f"{gh_id[0]}+{gh_user}@users.noreply.github.com"

!git config user.name {gh_user}
!git config user.email "{email}"

## 2. DagsHub: connect your fork

**Before continuing:** go to [dagshub.com](https://dagshub.com) and create the project from your new GitHub repository:
*Create → New repository → Clone a repository → GitHub*

Your DagsHub repository will then provide a DVC remote (data storage) and an MLflow tracking server.

## 3. Authenticate to DagsHub

The DagsHub client starts an OAuth login (follow the printed link) and caches a **token**.
This token is used as a password for both the DVC remote and the MLflow server, so you never write your password in the notebook.

The MLflow credentials are stored as **environment variables**: MLflow reads them automatically,
and every command launched from this notebook (e.g. `!bentoml serve`) inherits them.

In [ ]:
%pip install -q dagshub mlflow dvc

In [ ]:
import os
from dagshub.auth import get_token
from dagshub.common.api import UserAPI

token = get_token()                                  # OAuth login on first call, cached afterwards
dagshub_user = UserAPI.get_current_user().username   # may differ from your GitHub username

os.environ["DAGSHUB_USER_TOKEN"] = token
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{dagshub_user}/{repo_name}.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_user
os.environ["MLFLOW_TRACKING_PASSWORD"] = token

print("DagsHub user:", dagshub_user)

## 4. Configure DVC

Your fork contains the DVC configuration of the original repository: we point the `origin` remote to **your** DagsHub repository.
Credentials are written with `--local`, i.e. in `.dvc/config.local`, which is git-ignored: the token is never committed.

In [ ]:
!dvc remote add --default --force origin https://dagshub.com/{dagshub_user}/{repo_name}.dvc
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user {dagshub_user}
!dvc remote modify origin --local password {token}
!dvc config core.autostage true
!git add .dvc/config

## 5. Configure MLflow

MLflow already knows the tracking server through the environment variables set above.
We only select the experiment in which all runs of this notebook will be logged.

In [ ]:
import mlflow

print("Tracking URI:", mlflow.get_tracking_uri())

# set_experiment fails on a deleted experiment: restore it first
client = mlflow.MlflowClient()
exp = client.get_experiment_by_name("busi-classification")
if exp is not None and exp.lifecycle_stage == "deleted":
    client.restore_experiment(exp.experiment_id)

mlflow.set_experiment("busi-classification")

# Data Exploration

## Download the raw dataset
The raw dataset is exposed as a DagsHub Data Engine datasource on the original repository.
We first query the metadata, then download the files.

In [ ]:
from dagshub.data_engine import datasources

ds = datasources.get(f"{upstream_owner}/{repo_name}", "Dataset_BUSI_with_GT")

res = ds.all()                                           # query: metadata only
local_dir = res.download_files(os.getcwd() + "/data")    # actual download
print(local_dir)

## Let's take a look at our data

In [ ]:
RAW_DIR = local_dir / "Dataset_BUSI_with_GT"      # folder downloaded from the datasource

for root, dirs, files in os.walk(RAW_DIR):
    print("Current directory:", root)
    print("Subdirectories:", dirs)
    print("Files:", files)
    print('----------------')

# The data is there

Our dataset is made of 3 labels `['benign', 'normal', 'malignant']`, and each sample has 2 images: "main" and "mask". We won't use the "mask" images.

## Cleaning & formatting data

In [ ]:
import re

OUT_DIR = local_dir / "processed"
CLASSES = ["benign", "malignant", "normal"]
SEED = 42
VAL_SIZE, TEST_SIZE = 0.15, 0.15

MASK_RE = re.compile(r"_mask(_\d+)?$", re.IGNORECASE)   # "_mask", "_mask_1", ...

In [ ]:
import pandas as pd

# Inventory and exclusion
rows = []
for label in CLASSES:
    for p in sorted((RAW_DIR / label).glob("*.png")):
        rows.append({
            "path": p.as_posix(),
            "label": label,
            "is_mask": bool(MASK_RE.search(p.stem)),
        })

all_files = pd.DataFrame(rows)
print(all_files.groupby(["label", "is_mask"]).size().unstack(fill_value=0))

df = all_files[~all_files["is_mask"]].drop(columns="is_mask").reset_index(drop=True)
print(f"\n{len(df)} images kept out of {len(all_files)} files")

In [ ]:
import hashlib
from PIL import Image

# Duplicates
def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def img_meta(path):
    with Image.open(path) as im:
        return im.width, im.height, im.mode

df["md5"] = df["path"].map(file_md5)
df[["width", "height", "mode"]] = df["path"].map(img_meta).tolist()
display(df.head())

dup_groups = df[df.duplicated("md5", keep=False)].sort_values("md5")
print(f"{dup_groups['md5'].nunique()} duplicate groups, {len(dup_groups)} files involved")

# Duplicates with conflicting labels: report them and exclude them
conflicts = dup_groups.groupby("md5")["label"].nunique()
conflict_md5 = conflicts[conflicts > 1].index
print(f"{len(conflict_md5)} duplicates with conflicting labels (excluded)")
display(dup_groups.head(10))

df = df[~df["md5"].isin(conflict_md5)]
df = df.drop_duplicates("md5", keep="first").reset_index(drop=True)
print(f"{len(df)} unique images after deduplication")

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split
train_val, test = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["label"], random_state=SEED
)
train, val = train_test_split(
    train_val, test_size=VAL_SIZE / (1 - TEST_SIZE),
    stratify=train_val["label"], random_state=SEED,
)

df["split"] = "train"
df.loc[val.index, "split"] = "val"
df.loc[test.index, "split"] = "test"

summary = pd.crosstab(df["label"], df["split"], margins=True)
display(summary)
display(pd.crosstab(df["label"], df["split"], normalize="columns").round(3))

In [ ]:
# Export
OUT_DIR.mkdir(parents=True, exist_ok=True)
df["label_id"] = df["label"].map({c: i for i, c in enumerate(CLASSES)})
df = df[["path", "label", "label_id", "split", "width", "height", "mode", "md5"]]

df.to_csv(OUT_DIR / "index.csv", index=False)
dup_groups.to_csv(OUT_DIR / "duplicates_report.csv", index=False)
print(f"Index written: {OUT_DIR / 'index.csv'} ({len(df)} rows)")

## Version the data with DVC

`dvc add` creates small `.dvc` pointer files tracked by git (staged automatically thanks to `core.autostage`).
The data itself is pushed to your DagsHub DVC remote with `dvc push`.

In [ ]:
!dvc add {RAW_DIR} {OUT_DIR}
!git status

In [ ]:
!git commit -m "add processed data"
!git push
!dvc push

# Sklearn & XGBoost

## Data formatting

In [ ]:
IMG_SIZE = 224                       # torch (pretrained backbone)
SK_SIZE = 64                         # sklearn: small size, flattened vectors
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
BATCH_SIZE = 32

splits = {s: df[df["split"] == s].reset_index(drop=True) for s in ["train", "val", "test"]}
{s: len(d) for s, d in splits.items()}

In [ ]:
import numpy as np

def load_sklearn_array(frame, size=SK_SIZE):
    X = np.empty((len(frame), size * size), dtype=np.float32)
    for i, path in enumerate(frame["path"]):
        with Image.open(path) as im:
            im = im.convert("L").resize((size, size), Image.BILINEAR)
        X[i] = np.asarray(im, dtype=np.float32).ravel() / 255.0
    return X, frame["label_id"].to_numpy()

X_train, y_train = load_sklearn_array(splits["train"])
X_val,   y_val   = load_sklearn_array(splits["val"])
X_test,  y_test  = load_sklearn_array(splits["test"])

print(X_train.shape, X_val.shape, X_test.shape)

# Cache to avoid reloading the images at each run
np.savez_compressed(OUT_DIR / f"sklearn_{SK_SIZE}px.npz",
                    X_train=X_train, y_train=y_train,
                    X_val=X_val, y_val=y_val,
                    X_test=X_test, y_test=y_test)

## Train the sklearn model with mlflow tracking

In [ ]:
import mlflow
from sklearn.metrics import accuracy_score, f1_score, classification_report

INDEX_MD5 = hashlib.md5((OUT_DIR / "index.csv").read_bytes()).hexdigest()

def common_params(model_type):
    return {"model_type": model_type, "seed": SEED, "index_md5": INDEX_MD5,
            "n_train": len(splits["train"]), "n_val": len(splits["val"]),
            "n_test": len(splits["test"])}

def log_eval(y_true, y_pred, prefix):
    metrics = {f"{prefix}_accuracy": accuracy_score(y_true, y_pred),
               f"{prefix}_f1_macro": f1_score(y_true, y_pred, average="macro")}
    mlflow.log_metrics(metrics)
    return metrics

def log_report(y_true, y_pred):
    report = classification_report(y_true, y_pred, target_names=CLASSES)
    mlflow.log_text(report, "classification_report_test.txt")
    print(report)

In [ ]:
from sklearn.linear_model import LogisticRegression
import time

mlflow.sklearn.autolog(log_input_examples=True, log_datasets=False,
                       log_post_training_metrics=False,
                       serialization_format="skops")

with mlflow.start_run(run_name="sklearn_logreg"):
    mlflow.log_params({**common_params("sklearn_logreg"), "img_size": SK_SIZE})

    t0 = time.time()
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    clf.fit(X_train, y_train)
    mlflow.log_metric("train_time_s", time.time() - t0)

    log_eval(y_val, clf.predict(X_val), "val")
    y_pred = clf.predict(X_test)
    print(log_eval(y_test, y_pred, "test"))
    log_report(y_test, y_pred)

mlflow.sklearn.autolog(disable=True)

## Train the XGBoost model

In [ ]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
import time

mlflow.xgboost.autolog(log_input_examples=True, log_datasets=False)

with mlflow.start_run(run_name="xgboost"):
    mlflow.log_params({**common_params("xgboost"), "img_size": SK_SIZE})

    t0 = time.time()
    xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                        tree_method="hist", random_state=SEED, n_jobs=-1)
    xgb.fit(X_train, y_train,
            sample_weight=compute_sample_weight("balanced", y_train),
            eval_set=[(X_val, y_val)], verbose=False)   # params, mlogloss curve per iteration and model are logged
    mlflow.log_metric("train_time_s", time.time() - t0)

    log_eval(y_val, xgb.predict(X_val), "val")
    y_pred = xgb.predict(X_test)
    print(log_eval(y_test, y_pred, "test"))
    log_report(y_test, y_pred)

mlflow.xgboost.autolog(disable=True)

# PyTorch

## Data formatting

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

eval_tf = T.Compose([                # = the transform that will go into BentoML
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class BusiDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label_id"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as im:
            img = im.convert("RGB")
        return self.transform(img), self.labels[idx]

datasets_torch = {
    "train": BusiDataset(splits["train"], train_tf),
    "val":   BusiDataset(splits["val"], eval_tf),
    "test":  BusiDataset(splits["test"], eval_tf),
}

## Create the loader

In [ ]:
g = torch.Generator().manual_seed(SEED)

loaders = {
    s: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=(s == "train"),
                  num_workers=2, generator=g if s == "train" else None)
    for s, ds in datasets_torch.items()
}

# Imbalance benign >> malignant > normal: weights for CrossEntropyLoss
counts = splits["train"]["label_id"].value_counts().sort_index().to_numpy()
class_weights = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32)
print(dict(zip(CLASSES, class_weights.round(decimals=3).tolist())))

# Check one batch
xb, yb = next(iter(loaders["train"]))
print(xb.shape, xb.dtype, yb[:10])

## Load pretrained ResNet-18 and finetune it with early stopping

In [ ]:
import copy, os, time
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
MAX_EPOCHS, MIN_EPOCHS, PATIENCE, MIN_DELTA, LR = 50, 15, 8, 1e-3, 1e-3
os.environ["MLFLOW_DEFAULT_PREDICTION_DEVICE"] = "cpu"   # MLflow validates the logged model on CPU


def run_eval(model, loader, criterion):
    """Return (mean loss, y_true, y_pred) over a DataLoader."""
    model.eval()
    total_loss, labels, preds = 0.0, [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += criterion(logits, yb).item() * len(xb)
            labels.append(yb.cpu())
            preds.append(logits.argmax(1).cpu())
    return (total_loss / len(loader.dataset),
            torch.cat(labels).numpy(), torch.cat(preds).numpy())


with mlflow.start_run(run_name="torch_resnet18_es"):
    mlflow.log_params({**common_params("torch_resnet18"),
                       "max_epochs": MAX_EPOCHS, "min_epochs": MIN_EPOCHS,
                       "patience": PATIENCE, "min_delta": MIN_DELTA,
                       "early_stopping_metric": "val_loss",
                       "lr": LR, "batch_size": BATCH_SIZE, "img_size": IMG_SIZE,
                       "mean": MEAN, "std": STD, "frozen_backbone": True})

    # --- Model: pretrained ResNet-18, frozen backbone, new classification head ---
    torch.manual_seed(SEED)
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)

    best_loss, best_epoch, best_state = float("inf"), 0, None
    epochs_without_improvement = 0
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        # --- Training ---
        model.train()
        running = 0.0
        for xb, yb in loaders["train"]:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(xb)
        train_loss = running / len(loaders["train"].dataset)

        # --- Validation ---
        val_loss, y_true, y_hat = run_eval(model, loaders["val"], criterion)
        val_f1 = f1_score(y_true, y_hat, average="macro")
        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss,
                            "val_f1_macro_epoch": val_f1}, step=epoch)

        # --- Early stopping on validation loss ---
        if val_loss < best_loss - MIN_DELTA:
            best_loss, best_epoch = val_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
            flag = "*"
        else:
            epochs_without_improvement += 1
            flag = ""

        print(f"epoch {epoch:02d} | train_loss {train_loss:.3f} | "
              f"val_loss {val_loss:.3f} | val_f1 {val_f1:.3f} {flag}")

        if epoch >= MIN_EPOCHS and epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best epoch: {best_epoch})")
            break

    mlflow.log_metrics({"train_time_s": time.time() - t0,
                        "best_epoch": best_epoch, "stopped_epoch": epoch,
                        "best_val_loss": best_loss})

    # --- Restore best weights, then final evaluation ---
    model.load_state_dict(best_state)
    _, y_true, y_pred = run_eval(model, loaders["val"], criterion)
    log_eval(y_true, y_pred, "val")
    _, y_true, y_pred = run_eval(model, loaders["test"], criterion)
    print(log_eval(y_true, y_pred, "test"))
    log_report(y_true, y_pred)

    # --- Export as pt2 (traced graph, no pickle) ---
    model.cpu().eval()                                     # eval mode and CPU baked into the graph
    example = next(iter(loaders["test"]))[0][:4].numpy()   # batch > 1 keeps the batch dim dynamic
    mlflow.pytorch.log_model(
        model, name="model", serialization_format="pt2", input_example=example,
        pip_requirements=[f"torch=={torch.__version__.split('+')[0]}"],
    )

# Runs comparison

In [9]:
runs = mlflow.search_runs(experiment_names=["busi-classification"],
                          order_by=["metrics.test_f1_macro DESC"])
display(runs[["tags.mlflow.runName", "metrics.test_accuracy",
              "metrics.test_f1_macro", "metrics.train_time_s"]])

,tags.mlflow.runName,metrics.test_accuracy,metrics.test_f1_macro,metrics.train_time_s
0,torch_resnet18_es,0.760684,0.731962,249.356551
1,xgboost,0.786325,0.725066,145.076522
2,sklearn_logreg,0.692308,0.655574,17.454660


# BentoML

Our best model is now logged in MLflow, but a logged model is not a product yet: nobody can send it an ultrasound image and get a diagnosis back. This last step turns it into a **prediction service**.

[BentoML](https://docs.bentoml.com) is a framework to package and serve ML models. It fills the gap between the experiment tracker and the production infrastructure:

- **Service**: a Python class that loads the model once and exposes one or more **APIs** (HTTP endpoints). This is where the pre-processing lives: the image must go through exactly the same transform as during evaluation (`eval_tf`), otherwise the predictions are meaningless.
- **Bento**: the deployable artifact. It bundles the service code, the model and its dependencies (Python packages, system libraries) in a versioned, self-contained archive.
- **Container**: a Bento can be turned into a Docker image with a single command, and then deployed anywhere: a server, Kubernetes, or a cloud platform.

In this section, we will:

1. **Select** the best run from the MLflow comparison above and fetch its model from the DagsHub tracking server.
2. **Write** a BentoML service that takes an image, applies the pre-processing and returns the predicted class (`benign`, `malignant` or `normal`) with its probabilities.
3. **Serve** it locally and send it a test request.
4. **Build** a Bento and **containerize** it, ready for production.

> The service connects to the MLflow server through the `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME` and `MLFLOW_TRACKING_PASSWORD` environment variables set in the setup section. The code itself contains no credentials: in production, they are injected at runtime (e.g. `docker run -e ...`).

In [ ]:
# The MLflow credentials were set as environment variables in the setup section (step 3):
# commands launched from this notebook (e.g. `!bentoml serve`) inherit them.
for var in ["MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]:
    print(var, "=", "***" if var.endswith("PASSWORD") else os.environ.get(var))